# 03 SQL Business Metrics

This notebook converts the validted metrics from the EDA notebook into reusable SQL queries.

The purpose is to build SQL-based reporting logic that can be reused when new retail transaction data becomes available.

This SQL module focuses on reusable product-sales reporting metrics. Transaction-level net revenue and return/cancellation impact are analyzed in the EDA notebook.

Scope:
- Overall KPI metrics
- Monthly product sales metrics
- Country-level sales metrics
- Product-level sales metrics

In [1]:
import pandas as pd
import sqlite3

In [2]:
clean_product_sales = pd.read_csv("../data/processed/clean_product_sales.csv")
clean_all_transactions = pd.read_csv("../data/processed/clean_all_transactions.csv")

clean_product_sales.head()

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country,is_cancelled,revenue,is_negative_quantity,is_zero_quantity,is_positive_sale,is_non_product_transaction,invoice_year,invoice_month,invoice_day,invoice_hour,day_of_week
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday


In [3]:
conn = sqlite3.connect("../data/processed/retail_analytics.db")

clean_product_sales.to_sql(
    "clean_product_sales",
    conn,
    if_exists="replace",
    index=False
)

clean_all_transactions.to_sql(
    "clean_all_transactions",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite database created successfully.")

SQLite database created successfully.


In [4]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    conn
)
tables

,name
0,clean_product_sales
1,clean_all_transactions


## Overall KPI Metrics

This query calculates the main product sales KPIs, including product sales revenue, customers, average order value, and total quantity sold.

In [5]:
overall_kpi_query = """
SELECT
    ROUND(SUM(revenue), 2) AS product_sales_revenue,
    COUNT(DISTINCT invoice) AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers,
    ROUND(SUM(revenue) / COUNT(DISTINCT invoice), 2) AS average_order_value,
    SUM(quantity) AS total_quantity_sold
FROM clean_product_sales;
"""

overall_kpi_sql = pd.read_sql_query(overall_kpi_query, conn)

overall_kpi_sql

,product_sales_revenue,total_orders,total_customers,average_order_value,total_quantity_sold
0,19656007.51,39573,5861,496.7,11188413


## Monthly Product Sales Metrics

This query creates monthly product sales metrics for reporting and dashboard use.

In [6]:
monthly_sales_query = """
SELECT
    invoice_month,
    ROUND(SUM(revenue), 2) AS product_sales_revenue,
    COUNT(DISTINCT invoice) AS orders,
    COUNT(DISTINCT customer_id) AS customers,
    SUM(quantity) AS quantity_sold,
    ROUND(SUM(revenue) / COUNT(DISTINCT invoice), 2) AS average_order_value
FROM clean_product_sales
GROUP BY invoice_month
ORDER BY invoice_month;
"""

monthly_sales_sql = pd.read_sql_query(monthly_sales_query, conn)
monthly_sales_sql.head()

,invoice_month,product_sales_revenue,orders,customers,quantity_sold,average_order_value
0,2009-12,798232.03,1671,952,425252,477.70
1,2010-01,621353.43,1089,718,390471,570.57
2,2010-02,537926.69,1189,771,381621,452.42
3,2010-03,761748.53,1647,1051,525045,462.51
4,2010-04,646605.49,1438,939,366077,449.66


## Country-Level Sales Metrics

This query creates contry-level product sales metrics for reporting and dashboard use.

In [7]:
country_performance_query = """
SELECT
    country,
    Round(sum(revenue), 2) AS product_sales_revenue,
    COUNT(DISTINCT invoice) AS orders,
    COUNT(DISTINCT customer_id) AS customers,
    SUM(quantity) AS quantity_sold,
    ROUND(SUM(revenue) / COUNT(DISTINCT invoice), 2) AS average_order_value
FROM clean_product_sales
GROUP BY country
ORDER BY product_sales_revenue DESC;
"""

country_performance_sql = pd.read_sql_query(country_performance_query, conn)
country_performance_sql.head(15)

,country,product_sales_revenue,orders,customers,quantity_sold,average_order_value
0,United Kingdom,16810878.72,36217,5336,9176519,464.17
1,EIRE,625037.53,594,5,336101,1052.25
2,Netherlands,549773.41,216,22,383625,2545.25
3,Germany,383419.24,756,107,223091,507.17
4,France,311147.02,599,94,270583,519.44
5,Australia,167800.01,89,15,103753,1885.39
6,Spain,97994.50,147,41,49999,666.63
7,Switzerland,94024.59,85,22,52612,1106.17
8,Sweden,86319.14,99,19,88537,871.91
9,Denmark,67422.69,42,12,237406,1605.30


## Product-Level Sales Metrics

This query creates product-level sales metrics for indentifying top products by revenue, quantity, orders, and customer reach.

In [8]:
product_performance_query = """
SELECT
    stockcode,
    description,
    ROUND(SUM(revenue), 2) AS product_sales_revenue,
    SUM(quantity) AS quantity_sold,
    COUNT(DISTINCT invoice) AS orders,
    COUNT(DISTINCT customer_id) AS customers,
    ROUND(AVG(price), 2) AS average_unit_price
FROM clean_product_sales
GROUP BY stockcode, description
ORDER BY product_sales_revenue DESC, stockcode ASC, description ASC;
"""

product_performance_sql = pd.read_sql_query(product_performance_query, conn)
product_performance_sql.head(15)

,stockcode,description,product_sales_revenue,quantity_sold,orders,customers,average_unit_price
0,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918,1314,14.19
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356,1490,3.08
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1,2.08
3,47566,PARTY BUNTING,148318.28,28200,2674,894,5.72
4,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245,860,2.37
5,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807,1010,1.86
6,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,2018,896,3.37
7,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247,138,1.47
8,79321,CHILLI LIGHTS,80540.88,15841,1135,304,6.29
9,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,71300.40,31409,756,300,3.34


## SQL Business Metrics Summary

This notebook converted the validated metrics from the EDA notebook into reusable SQL queries.

The SQL queries generate reporting-ready tables for:
- Overall KPI metrics
- Monthly product sales metrics
- Country-level sales metrics
- Product-level sales metrics

These queries can be reused when new cleaned transaction data becomes available.

In [9]:
overall_kpi_sql.to_csv("../data/processed/sql_overall_kpi.csv", index=False)
monthly_sales_sql.to_csv("../data/processed/sql_monthly_sales.csv", index=False)
country_performance_sql.to_csv("../data/processed/sql_country_performance.csv", index=False)
product_performance_sql.to_csv("../data/processed/sql_product_performance.csv", index=False)

print("SQL metric tables exported successfully.")

SQL metric tables exported successfully.


In [10]:
sql_script = f"""
-- 03 SQL Business Metrics
-- This file contains reusable SQL queries for retail business reporting.

-- 1. Overall KPI Metrics
{overall_kpi_query}

-- 2. Monthly Product Sales Metrics
{monthly_sales_query}

-- 3. Country Performance Metrics
{country_performance_query}

-- 4. Product-Level Sales Metrics
{product_performance_query}
"""

with open("../sql/01_business_metrics.sql", "w") as file:
    file.write(sql_script)

print("SQL script file created successfully.")

SQL script file created successfully.


In [11]:
conn.close()